# Feature Engineering

1. Extract initial conditions from each cleaned external
source
2. Apply transformations, merge into a single 6-feature DataFrame, and
3. save as wbl_features.csv.

**Sources:**

1. Hyland, Djankov & Goldberg (2020, AER: Insights)
2. Marshall, Gurr & Jaggers (2019, Polity5 Codebook)
3. Chen & Guestrin (2016, KDD)
4. Molnar (2025, Ch. 9)

**Saves:**
1. data/processed/wbl_features.csv (Matrix B — 189 x 6+)
2. data/external/baseline_years_used.csv   (appendix table)
3. outputs/figures/matrix_b_distributions.png

In [ ]:
# Change this path to match yours
BASE = '/content/drive/MyDrive/wbl_project'

In [ ]:
# uncomment both lines if you want to connect this notebook to Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Download all libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Lock economy list from Matrix A — all feature extraction is anchored to this
panel = pd.read_csv(f'{BASE}/data/processed/wbl_panel_final.csv', index_col=0)
ECONOMIES = panel.index.tolist()

print(f'Economy list locked: {len(ECONOMIES)} ISO3 codes')
print(f'BASE exists: {os.path.exists(BASE)}')

Mounted at /content/drive
Economy list locked: 189 ISO3 codes
BASE exists: True


In [ ]:
# Create a file for utility functions
# For a given WDI raw file, extracts for each economy in ECONOMIES:
#   - the value at base_year (1971 by default)
#   - if missing, the earliest available year (fallback)
#   - the actual year used (recorded for the appendix table)
#
# This implements DD-004b. Following Hyland, Djankov & Goldberg (2020,
# AER: Insights): use earliest available observation, disclose in appendix.

def extract_initial_condition(fname, value_col_name, base_year=1971):
    """
    Load a WDI raw CSV and extract one value per economy.

    Parameters
    ----------
    fname          : filename in data/external/ (e.g. 'gdp_pc_raw.csv')
    value_col_name : name to assign the extracted value column
    base_year      : target year (default 1971)

    Returns
    -------
    DataFrame with columns: economy, <value_col_name>, <value_col_name>_year
    """
    path = f'{BASE}/data/external/{fname}'
    df   = pd.read_csv(path)

    # Keep only panel economies
    df = df[df['economy'].isin(ECONOMIES)].copy()

    # Identify year columns
    yr_cols = sorted([c for c in df.columns if c.startswith('YR')],
                     key=lambda x: int(x[2:]))

    rows = []
    for eco in ECONOMIES:
        sub = df[df['economy'] == eco]

        if sub.empty:
            rows.append({'economy': eco,
                         value_col_name: np.nan,
                         f'{value_col_name}_year': np.nan})
            continue

        sub = sub.iloc[0]

        # 1. Try exact base year
        base_col = f'YR{base_year}'
        if base_col in sub.index and pd.notna(sub[base_col]):
            rows.append({'economy': eco,
                         value_col_name: sub[base_col],
                         f'{value_col_name}_year': base_year})
            continue

        # 2. Fallback: earliest available year
        valid = [(int(c[2:]), sub[c]) for c in yr_cols if pd.notna(sub[c])]
        if valid:
            valid.sort()
            yr, val = valid[0]
            rows.append({'economy': eco,
                         value_col_name: val,
                         f'{value_col_name}_year': yr})
        else:
            rows.append({'economy': eco,
                         value_col_name: np.nan,
                         f'{value_col_name}_year': np.nan})

    result = pd.DataFrame(rows)
    return result


def summarise_baseline(df, label):
    """Print baseline year distribution for a feature."""
    yr_col = f'{label}_year'
    years  = df[yr_col].dropna()
    n_1971    = (years == 1971).sum()
    n_pre90   = ((years > 1971) & (years <= 1990)).sum()
    n_post90  = (years > 1990).sum()
    n_missing = df[yr_col].isna().sum()
    print(f'{label}:')
    print(f'  At exact 1971:      {n_1971}')
    print(f'  Fallback 1972-1990: {n_pre90}')
    print(f'  Fallback > 1990:    {n_post90}')
    print(f'  Missing entirely:   {n_missing}')
    if n_post90 > 0:
        late = df[df[yr_col] > 1990][['economy', yr_col]]
        print(f'  Late fallback list: {late["economy"].tolist()}')
    print()


print('extract_initial_condition() defined')

✅ extract_initial_condition() defined


In [ ]:
# log-transformed GDP per capita
#
# Log transformation applied because:
#   - GDP per capita is right-skewed across countries
#   - The economic literature (Hyland et al. 2020) uses log GDP
#     to reflect that marginal gains at low income have larger
#     real-world effects than equal gains at high income
#   - Reduces leverage of extreme outliers (Gulf states, Luxembourg)
#
# Expected: ~136 economies at exact 1971

gdp = extract_initial_condition('gdp_pc_raw.csv', 'gdp_pc')
summarise_baseline(gdp, 'gdp_pc')

# Log-transform: clip at 1 to avoid log(0) for any anomalous zeros
gdp['gdp_pc_log'] = np.log(gdp['gdp_pc'].clip(lower=1))

print('GDP per capita — value range before log:')
print(f'Min: {gdp["gdp_pc"].min():.1f}  Max: {gdp["gdp_pc"].max():.1f}')
print(f'After log — Min: {gdp["gdp_pc_log"].min():.2f}  Max: {gdp["gdp_pc_log"].max():.2f}')
print(f'Missing after extraction: {gdp["gdp_pc_log"].isna().sum()}')

gdp_pc:
  At exact 1971:      136
  Fallback 1972-1990: 41
  Fallback > 1990:    12
  Missing entirely:   0
  Late fallback list: ['PSE', 'SSD', 'SRB', 'SMR', 'MOZ', 'MNE', 'LTU', 'LVA', 'XKX', 'EST', 'ERI', 'AFG']

GDP per capita — value range before log:
  Min: 21.1  Max: 37853.0
  After log — Min: 3.05  Max: 10.54
  Missing after extraction: 0


In [ ]:
# FLFP (Female Labour Force Participation)
# Source: WDI indicator SL.TLF.CACT.FE.ZS (ILO modelled estimate)
# Unit: % of female population aged 15+ economically active.
# DISCLOSURE (DD-005): Zero economies have FLFP at exact 1971.
# All values are fallback years. The ILO modelled estimates begin
# from the late 1980s for most economies.
# Following Hyland, Djankov & Goldberg (2020): use earliest available,
# report distribution of years used in the paper appendix.
# No transformation applied: FLFP is already on a bounded 0-100 scale
# comparable across economies.

flfp = extract_initial_condition('flfp_raw.csv', 'flfp')
summarise_baseline(flfp, 'flfp')

# Full distribution of years used — this becomes Table A1 in the paper
yr_dist = flfp['flfp_year'].value_counts().sort_index()
print('Distribution of years used for FLFP initial condition:')
print(yr_dist.to_string())
print()
print(f'Median year used: {flfp["flfp_year"].median():.0f}')
print(f'Missing entirely: {flfp["flfp"].isna().sum()}')

flfp:
  At exact 1971:      0
  Fallback 1972-1990: 178
  Fallback > 1990:    0
  Missing entirely:   11

Distribution of years used for FLFP initial condition:
flfp_year
1990.0    178

Median year used: 1990
Missing entirely: 11


In [ ]:
# Trade openness
# Source: WDI indicator NE.TRD.GNFS.ZS
# No transformation needed. Values above 100% are real —
# small open economies (Singapore, Malta) have trade volumes
# exceeding GDP. These are structural outliers documented in DD-005.
# DTW-PAM is robust to outliers in features per Maharaj et al. (2019 Ch.2).
#
# Expected: 95 at exact 1971, 38 fallback after 1990, 14 missing
trade = extract_initial_condition('trade_openness_raw.csv', 'trade_openness')
summarise_baseline(trade, 'trade_openness')

print('Trade openness — value range:')
print(f'  Min: {trade["trade_openness"].min():.1f}')
print(f'  Max: {trade["trade_openness"].max():.1f}')
print(f'  Economies above 150%: {(trade["trade_openness"] > 150).sum()}')
if (trade["trade_openness"] > 150).sum() > 0:
    print(trade[trade["trade_openness"] > 150][["economy","trade_openness","trade_openness_year"]])

trade_openness:
  At exact 1971:      95
  Fallback 1972-1990: 42
  Fallback > 1990:    38
  Missing entirely:   14
  Late fallback list: ['ZMB', 'PSE', 'UZB', 'ARE', 'TLS', 'TJK', 'SUR', 'SSD', 'SRB', 'SMR', 'WSM', 'QAT', 'POL', 'PLW', 'MOZ', 'MNE', 'MDA', 'MHL', 'MDV', 'MWI', 'LTU', 'LVA', 'XKX', 'KAZ', 'HUN', 'ETH', 'SWZ', 'EST', 'ERI', 'GNQ', 'DJI', 'HRV', 'COD', 'KHM', 'BIH', 'ATG', 'AGO', 'AFG']

Trade openness — value range:
  Min: 4.9
  Max: 347.7
  Economies above 150%: 8
    economy  trade_openness  trade_openness_year
19      TLS      175.648460               2000.0
39      SGP      259.296671               1971.0
46      SMR      321.364013               2015.0
85      MDV      165.979251               2014.0
89      LUX      168.222016               1971.0
116     HKG      174.509951               1971.0
143     DJI      347.720317               2013.0
177     BHR      210.498536               1980.0


In [ ]:
# Income group and region
# Source: WDI metadata (wdi_metadata_raw.csv)
# These are stable World Bank classifications
# Income group encoding:
#  Ordinal 1-4: LIC=1, LMC=2, UMC=3, HIC=4
#  Reflects the ordered development continuum in Al-Marhubi (2026) S.2.2.
#  Two economies were manually assigned — verified in data_exploration.ipynb.
#
# Region encoding (DD-006):
# One-hot encoded (7 binary columns, one per region).
# Region is nominal — no natural ordering exists between EAS, ECS, LCN etc.
# One-hot encoding prevents XGBoost from inferring spurious ordinal
# relationships. Each region gets its own SHAP value for interpretation.
# Source: Celik & Koroglu (2026); Molnar (2025, Ch. 9).

meta = pd.read_csv(f'{BASE}/data/external/wdi_metadata_raw.csv')
meta = meta[meta['economy'].isin(ECONOMIES)].copy()

income_map = {'LIC': 1, 'LMC': 2, 'UMC': 3, 'HIC': 4}
meta['income_group'] = meta['incomeLevel'].map(income_map)

print('Income group distribution:')
print(meta['incomeLevel'].value_counts().to_string())
print(f'Missing after encoding: {meta["income_group"].isna().sum()}')

if meta['income_group'].isna().sum() > 0:
    print('\n⚠️  Remaining missing income groups:')
    print(meta[meta['income_group'].isna()][['economy','name','incomeLevel']])
    print('Resolve before proceeding.')
else:
    print('\n✅ All income groups assigned')

# Region one-hot encoding
region_dummies = pd.get_dummies(meta['region'], prefix='region')
meta_encoded   = pd.concat([meta[['economy','income_group']], region_dummies], axis=1)

print(f'\nRegion columns created: {[c for c in meta_encoded.columns if c.startswith("region_")]}')
print(f'Missing regions: {meta["region"].isna().sum()}')

Income group distribution:
incomeLevel
HIC    62
UMC    51
LMC    50
LIC    24
INX     2
Missing after encoding: 2

⚠️  Remaining missing income groups:
    economy           name incomeLevel
61      ETH       Ethiopia         INX
206     VEN  Venezuela, RB         INX
Resolve before proceeding.

Region columns created: ['region_EAS', 'region_ECS', 'region_LCN', 'region_MEA', 'region_NAC', 'region_SAS', 'region_SSF']
Missing regions: 0


In [ ]:
# Fix INX income groups — manual assignment (DD-005)
# ETH: LIC (last classified before INX suspension)
# VEN: UMC (last classified before INX suspension)
MANUAL_INCOME = {'ETH': 'LIC', 'VEN': 'UMC'}

for iso, group in MANUAL_INCOME.items():
    meta.loc[meta['economy'] == iso, 'incomeLevel'] = group
    print(f'Assigned {iso} -> {group}')

# Re-encode after manual fix
meta['income_group'] = meta['incomeLevel'].map(income_map)
meta_encoded = pd.concat([meta[['economy','income_group']],
                          pd.get_dummies(meta['region'], prefix='region')],
                         axis=1)

print(f'Missing after fix: {meta["income_group"].isna().sum()}')

Assigned ETH -> LIC
Assigned VEN -> UMC
Missing after fix: 0


In [ ]:
meta  = pd.read_csv(f'{BASE}/data/external/wdi_metadata_raw.csv')
panel = pd.read_csv(f'{BASE}/data/processed/wbl_panel_final.csv', index_col=0)
meta_panel = meta[meta['economy'].isin(panel.index)]
print(f"Missing income groups in saved CSV: {meta_panel['incomeLevel'].isna().sum()}")
print(meta_panel[meta_panel['incomeLevel'].isna()][['economy','name','incomeLevel']])

Missing income groups in saved CSV: 0
Empty DataFrame
Columns: [economy, name, incomeLevel]
Index: []


In [ ]:
# Democracy score (Polity V)
# ─────────────────────────────────────────────────────────
# Source: Polity V polity2 score (-10 to +10)
# Unit: integer, -10 = full autocracy, +10 = full democracy
#
# Three-part extraction (DD-005):
#   1. Exact 1971 value if available
#   2. Earliest available year if 1971 missing (fallback)
#   3. Score of 0 for economies absent from Polity V entirely
#      (small states below 500,000 population threshold)
#      Per Marshall, Gurr & Jaggers (2019, Polity5 Codebook):
#      standard default for unclassified small states.
#
# Note: polity5_raw.csv was generated in data_preprocessing.ipynb
# with ISO3 crosswalk already applied.

polity_path = f'{BASE}/data/external/polity5_raw.csv'
polity      = pd.read_csv(polity_path)

# Remove residual special codes if any remain
polity = polity[~polity['polity2'].isin([-66, -77, -88])].copy()

rows = []
for eco in ECONOMIES:
    sub = polity[polity['economy'] == eco].copy()

    if sub.empty:
        # Economy absent from Polity V — assign 0 (small state default)
        rows.append({'economy': eco, 'democracy': 0.0,
                     'democracy_year': np.nan, 'democracy_source': 'default_0'})
        continue

    # Try exact 1971
    at_1971 = sub[sub['year'] == 1971]
    if not at_1971.empty and pd.notna(at_1971.iloc[0]['polity2']):
        rows.append({'economy': eco,
                     'democracy': at_1971.iloc[0]['polity2'],
                     'democracy_year': 1971,
                     'democracy_source': 'exact_1971'})
        continue

    # Fallback: earliest available year
    sub_valid = sub.dropna(subset=['polity2']).sort_values('year')
    if not sub_valid.empty:
        first = sub_valid.iloc[0]
        rows.append({'economy': eco,
                     'democracy': first['polity2'],
                     'democracy_year': first['year'],
                     'democracy_source': 'fallback'})
    else:
        rows.append({'economy': eco, 'democracy': 0.0,
                     'democracy_year': np.nan, 'democracy_source': 'default_0'})

dem = pd.DataFrame(rows)

# Summary
n_exact = (dem['democracy_source'] == 'exact_1971').sum()
n_fallback= (dem['democracy_source'] == 'fallback').sum()
n_default = (dem['democracy_source'] == 'default_0').sum()

print('Democracy score extraction:')
print(f'At exact 1971: {n_exact}')
print(f'Fallback year: {n_fallback}')
print(f'Default 0 (absent): {n_default}')
print(f'Total: {len(dem)}')
print(f'Score range: {dem["democracy"].min()} to {dem["democracy"].max()}')
print()
if n_fallback > 0:
    summarise_baseline(dem.rename(columns={'democracy_year':'democracy_year'}),
                       'democracy')

Democracy score extraction:
  At exact 1971:       115
  Fallback year:       36
  Default 0 (absent):  38
  Total:               189

  Score range: -10.0 to 10.0

democracy:
  At exact 1971:      115
  Fallback 1972-1990: 14
  Fallback > 1990:    16
  Missing entirely:   38
  Late fallback list: ['UZB', 'UKR', 'TJK', 'SSD', 'SVN', 'SVK', 'MKD', 'KGZ', 'KAZ', 'ERI', 'CZE', 'HRV', 'BIH', 'BLR', 'AZE', 'ARM']



In [ ]:
# Merge all features into Matrix B 'economy' using left join
#
# Final columns in Matrix B:
#   gdp_pc_log         : log(GDP per capita, USD)
#   flfp               : FLFP % (fallback year — disclosed)
#   trade_openness     : trade % GDP
#   income_group       : ordinal 1-4
#   region_EAS ... SSF : 7 binary columns
#   democracy          : Polity V polity2 score

# Start from the full economy list
base_df = pd.DataFrame({'economy': ECONOMIES})

# Merge features one by one
matrix_b = (base_df
    .merge(gdp[['economy','gdp_pc_log']], on='economy', how='left')
    .merge(flfp[['economy','flfp']],      on='economy', how='left')
    .merge(trade[['economy','trade_openness']], on='economy', how='left')
    .merge(meta_encoded,                  on='economy', how='left')
    .merge(dem[['economy','democracy']],  on='economy', how='left')
)

matrix_b = matrix_b.set_index('economy')

print('MISSING VALUES PER FEATURE')
print(matrix_b.isnull().sum().to_string())

print('FEATURE RANGES')
print(matrix_b.describe().round(2).to_string())

=== MATRIX B SHAPE ===
Economies: 189
Features:  12
Columns:   ['gdp_pc_log', 'flfp', 'trade_openness', 'income_group', 'region_EAS', 'region_ECS', 'region_LCN', 'region_MEA', 'region_NAC', 'region_SAS', 'region_SSF', 'democracy']

=== MISSING VALUES PER FEATURE ===
gdp_pc_log         0
flfp              11
trade_openness    14
income_group       0
region_EAS         0
region_ECS         0
region_LCN         0
region_MEA         0
region_NAC         0
region_SAS         0
region_SSF         0
democracy          0

=== FEATURE RANGES ===
       gdp_pc_log    flfp  trade_openness  income_group  democracy
count      189.00  178.00          175.00        189.00     189.00
mean         6.43   47.86           69.06          2.80      -1.34
std          1.22   17.63           49.60          1.04       6.40
min          3.05    8.51            4.95          1.00     -10.00
25%          5.58   35.94           36.82          2.00      -7.00
50%          6.21   47.83           57.20          3.00

In [ ]:
# CELL 8 — Missing value handling decision
# ─────────────────────────────────────────────────────────
# XGBoost (Chen & Guestrin 2016) includes a sparsity-aware
# split-finding algorithm that learns the optimal default
# direction for missing values at each tree node. Missing
# values are handled natively — no imputation is required
# or appropriate before XGBoost.
#
# Imputing with regional medians would:
#   (1) replace genuine absence of information with synthetic values
#   (2) cause XGBoost to treat synthetic values as observed data
#   (3) introduce MNAR bias — missing economies are structurally
#       different from the regional majority used for imputation
#       (Little & Rubin 2019, Ch. 1)
#
# Decision (DD-006): no imputation applied to continuous features.
# Missing values pass through to Matrix B as NaN.
# XGBoost handles them natively in Day 4.

# Confirm income group is complete
assert matrix_b['income_group'].isna().sum() == 0, \
    'Income group has missing values — resolve before saving.'


=== MISSING VALUE AUDIT BEFORE SAVING ===

gdp_pc_log         0
flfp              11
trade_openness    14
income_group       0
region_EAS         0
region_ECS         0
region_LCN         0
region_MEA         0
region_NAC         0
region_SAS         0
region_SSF         0
democracy          0

Total missing cells: 25

✅ Income group: complete
✅ All other missing values: passed through as NaN for XGBoost

Decision recorded in DD-006: no imputation.
Reference: Chen & Guestrin (2016, KDD); Little & Rubin (2019, Ch. 1)


In [ ]:
# Save Matrix B and appendix table

# Save Matrix B
matrix_b_path = f'{BASE}/data/processed/wbl_features.csv'
matrix_b.to_csv(matrix_b_path)
print(f'Matrix B saved: {matrix_b_path}')
print(f'Shape: {matrix_b.shape}')
print(f'Columns: {matrix_b.columns.tolist()}')
print(f'Missing values: {matrix_b.isnull().sum().sum()}')

# Build appendix table of baseline years
app = pd.DataFrame({'economy': ECONOMIES})
for feat_df, col in [(gdp,'gdp_pc'), (flfp,'flfp'), (trade,'trade_openness')]:
    yr_col = f'{col}_year'
    app = app.merge(feat_df[['economy', yr_col]], on='economy', how='left')

app = app.merge(dem[['economy','democracy_year','democracy_source']],
                on='economy', how='left')
app = app.set_index('economy')

app_path = f'{BASE}/data/external/baseline_years_used.csv'
app.to_csv(app_path)
print(f' Appendix table saved: {app_path}')

# Summary for paper
print('APPENDIX TABLE SUMMARY')
for col in ['gdp_pc_year','flfp_year','trade_openness_year','democracy_year']:
    yr = app[col].dropna()
    n_1971   = (yr == 1971).sum()
    n_other  = (yr != 1971).sum()
    n_miss   = app[col].isna().sum()
    label    = col.replace('_year','')
    print(f'{label}: {n_1971} at 1971 | {n_other} fallback | {n_miss} absent/default')

✅ Matrix B saved: /content/drive/MyDrive/wbl_project/data/processed/wbl_features.csv
   Shape: (189, 12)
   Columns: ['gdp_pc_log', 'flfp', 'trade_openness', 'income_group', 'region_EAS', 'region_ECS', 'region_LCN', 'region_MEA', 'region_NAC', 'region_SAS', 'region_SSF', 'democracy']
   Missing values: 25

✅ Appendix table saved: /content/drive/MyDrive/wbl_project/data/external/baseline_years_used.csv

=== APPENDIX TABLE SUMMARY ===
gdp_pc: 136 at 1971 | 53 fallback | 0 absent/default
flfp: 0 at 1971 | 178 fallback | 11 absent/default
trade_openness: 95 at 1971 | 80 fallback | 14 absent/default
democracy: 115 at 1971 | 36 fallback | 38 absent/default


In [ ]:
# Record DD-006 in DESIGN_DECISIONS.md

dd_path = f'{BASE}/reports/DESIGN_DECISIONS.md'

# Get actual counts for the DD entry
region_cols = [c for c in matrix_b.columns if c.startswith('region_')]
dem_default = (dem['democracy_source'] == 'default_0').sum()
dem_fallback= (dem['democracy_source'] == 'fallback').sum()
dem_exact   = (dem['democracy_source'] == 'exact_1971').sum()

entry = f"""
## DD-006 — Matrix B Feature Engineering Decisions

**File:** data/processed/wbl_features.csv
**Shape:** {matrix_b.shape[0]} economies x {matrix_b.shape[1]} columns
**Missing values:** {matrix_b.isnull().sum().sum()}

### Feature list

1. gdp_pc_log — log(GDP per capita, current USD), WDI NY.GDP.PCAP.CD
   Transformation: natural log (Hyland et al. 2020)
   Baseline: {(gdp['gdp_pc_year'] == 1971).sum()} at exact 1971

2. flfp — FLFP %, WDI SL.TLF.CACT.FE.ZS
   Baseline: 0 at exact 1971. All values are fallback years.
   Median fallback year: {flfp['flfp_year'].median():.0f}
   Disclosure: Table A1 in paper appendix (baseline_years_used.csv)

3. trade_openness — trade % GDP, WDI NE.TRD.GNFS.ZS
   Baseline: {(trade['trade_openness_year'] == 1971).sum()} at exact 1971
   Outliers above 150% documented, not removed (DTW-PAM robust per Maharaj et al. 2019 Ch.2)

4. income_group — ordinal 1-4 (LIC=1, LMC=2, UMC=3, HIC=4)
   Source: WDI metadata. 2 manual assignments (DD-005).

5. region_* — 7 binary columns (one-hot encoded)
   Regions: {[c.replace('region_','') for c in region_cols]}
   Encoding justification: nominal variable, no natural ordering.
   Reference: Celik & Koroglu (2026); Molnar (2025, Ch.9)

6. democracy — Polity V polity2 score (-10 to +10)
   At exact 1971: {dem_exact}
   Fallback year:  {dem_fallback}
   Default 0 (absent from Polity V): {dem_default}
   Reference for default 0: Marshall, Gurr & Jaggers (2019, Polity5 Codebook)

### Excluded features
- Women in parliament: DD-004f (0/189 at 1971, all fallback >1990)
- CEDAW ratification year: DD-004e (13/189 unmappable)
- Gender-marked ODA: DD-004 (OECD CRS not systematic before 2002)
- Hofstede Individualism: DD-004c (76/189 coverage)

### Anti-leakage rule
No WBL sub-index information enters Matrix B.
Matrix A (WBL panel) and Matrix B (external features) are built
from entirely separate sources.

---
"""

with open(dd_path, 'a') as f:
    f.write(entry)

print('✅ DD-006 recorded in DESIGN_DECISIONS.md')
print()
print('=== MATRIX B READY ===')
print(f'Path:    data/processed/wbl_features.csv')
print(f'Shape:   {matrix_b.shape}')
print(f'Missing: {matrix_b.isnull().sum().sum()}')
print()
print('Next step: read Maharaj et al. (2019) Chapters 2-3')
print('then open clustering.ipynb')

✅ DD-006 recorded in DESIGN_DECISIONS.md

=== MATRIX B READY ===
Path:    data/processed/wbl_features.csv
Shape:   (189, 12)
Missing: 25

Next step: read Maharaj et al. (2019) Chapters 2-3
then open clustering.ipynb
